# Testing the Performance of HypEx Functions

## Imports

In [1]:
import numpy as np
import pandas as pd
import time
import tracemalloc
from itertools import product

from typing import Dict, Union, Any, Optional, TypedDict, List

from hypex import (
    AATest,
    ABTest,
    Matching,
    HomogeneityTest,
)

from hypex.dataset import (
    Dataset,
    InfoRole,
    ExperimentData,
    TreatmentRole,
    TargetRole,
    StratificationRole,
)

## PerformanceTester
### Performance Testing: Measuring Time and Memory & Synthetic Data Generator

In [2]:
class PerformanceTester:

    class DataParams(TypedDict):
        n_columns: List[int]
        n_rows: List[int]
        n2c_ratio: List[float]
        rs: Union[int, None]
        num_range: Union[tuple, None]
        n_categories: Union[int, None]

        default_params = {
            'n_columns': [100],
            'n_rows': [1000],
            'n2c_ratio': [0.5],
            'rs': 42,
            'num_range': (-100, 100),
            'n_categories': 5
        }

    def __init__(
        self,
        experiment: Union[AATest, ABTest, Matching, HomogeneityTest],
        data_params: Union[DataParams, None],
        experiment_params: Dict,
        use_time: bool=True,
        use_memory: bool=True
    ):
        self.experiment = experiment
        self.data_params = data_params
        [self.data_params.setdefault(k, v) for (k, v) in self.DataParams.default_params.items()]
        self.experiment_params = experiment_params
        self.use_time = use_time
        self.use_memory = use_memory
        self.resume = {}
    
    @staticmethod
    def _generate_synthetic_data(
        n_columns: int = 1000,
        n_rows: int = 1000,
        n2c_ratio: float = 0.5,
        rs: int = None,
        num_range: tuple = (1, 99),  # Define range for integer values
        n_categories = 5
    ) -> pd.DataFrame:
        """Creates data for tests.

        Args:
            n_columns: number of columns
            n_rows: number of rows
            n2c_ratio: ratio of numerical to categorical columns
            rs: random state for np
            num_range: range for integer values (inclusive, exclusive)

        Returns:
            data: DataFrame with the generated test data
        """

        if rs is not None:
            np.random.seed(rs)

        n_numerical = int(n_columns * n2c_ratio)
        n_categorical = n_columns - n_numerical

        numerical_data = np.random.randint(
            num_range[0], num_range[1], size=(n_rows, n_numerical)
        )

        categories = [f"Category_{i+1}" for i in range(n_categories)]
        categorical_data = np.random.choice(categories, size=(n_rows, n_categorical))

        data = pd.DataFrame(
            np.hstack((numerical_data, categorical_data)),
            columns=[f"num_col_{i}" for i in range(n_numerical)]
            + [f"cat_col_{i}" for i in range(n_categorical)],
        )

        return data
    
    def _create_dataset(self, params: List) -> Dataset:
        data = self._generate_synthetic_data(*params)
        return Dataset(
            roles={
                data.columns[0]: TreatmentRole(int),
                data.columns[1]: TargetRole(int),
                data.columns[-1]: TargetRole(),
            },
            data=data
        )

    def execute(self):
        combinations = list(product(*[
            param if isinstance(param, list) else [param] for _, param in self.data_params.items()
        ]))
        for combination in combinations:
            print(*[f'{list(self.DataParams.default_params.keys())[i]}: {combination[i]}' for i in range(len(combination))], sep='\n') #log
            data = self._create_dataset(combination)
            experiment = self.experiment(**self.experiment_params)
            res = self.test_function_performance(experiment.execute, param_dict={'data': data})
            self.resume[combination] = res
            
    def test_function_performance(self, func, param_dict):
        """
        Tests the performance of the specified function.

        :param func: The function to test
        :param args: Positional arguments for the function
        :param use_time: If True, measures execution time
        :param use_memory: If True, measures memory usage
        :param kwargs: Keyword arguments for the function
        :return: The result of the function execution
        """
        param_dict = param_dict or {}
        exec_time = None
        memory_usage = None

        if self.use_time:
            start_time = time.time()

        if self.use_memory:
            tracemalloc.start()

        result = func(**param_dict)

        if self.use_memory:
            _, memory_usage = tracemalloc.get_traced_memory()
            tracemalloc.stop()

        if self.use_time:
            end_time = time.time()
            exec_time = end_time - start_time

        if self.use_time:
            print(f"Execution time of '{func.__name__}': {exec_time:.4f} seconds") #log

        if self.use_memory:
            print(f"Memory usage of '{func.__name__}': {(memory_usage / 10**6):.4f} MB") #log

        return(
            result,
            exec_time if self.use_time else None,
            memory_usage if self.use_memory else None
        )

In [3]:
tester = PerformanceTester(
    experiment=AATest,
    data_params={
        'n_columns': [10**n for n in range(1, 5)],
        'n_rows': [10**n for n in range(3, 6)],
        'n2c_ratio': [0.5, 1],
    },
    experiment_params={
        'n_iterations': 10,
    },
    use_time=True,
    use_memory=True

)
tester.execute()

n_columns: 10
n_rows: 1000
n2c_ratio: 0.5
rs: 42
num_range: (-100, 100)
n_categories: 5
Execution time of 'execute': 9.8307 seconds
Memory usage of 'execute': 1.2484 MB
n_columns: 10
n_rows: 1000
n2c_ratio: 1
rs: 42
num_range: (-100, 100)
n_categories: 5
Execution time of 'execute': 8.6909 seconds
Memory usage of 'execute': 0.8549 MB
n_columns: 10
n_rows: 10000
n2c_ratio: 0.5
rs: 42
num_range: (-100, 100)
n_categories: 5
Execution time of 'execute': 13.1439 seconds
Memory usage of 'execute': 2.4422 MB
n_columns: 10
n_rows: 10000
n2c_ratio: 1
rs: 42
num_range: (-100, 100)
n_categories: 5
Execution time of 'execute': 12.9901 seconds
Memory usage of 'execute': 2.7437 MB
n_columns: 10
n_rows: 100000
n2c_ratio: 0.5
rs: 42
num_range: (-100, 100)
n_categories: 5


KeyboardInterrupt: 

### AA testing (random split testing)

In [41]:
data = create_test_data(8, 1000)
data = Dataset(
    roles={
        data.columns[0]: TreatmentRole(int),
        data.columns[1]: TargetRole(int),
        data.columns[-1]: TargetRole(),
        data.columns[2]: StratificationRole(str)
    },
    data=data,
)
data

NameError: name 'create_test_data' is not defined

In [6]:
aa = AATest(n_iterations=10)
res = test_function_performance(aa.execute, param_dict={'data': data})


Execution time of 'execute': 9.2285 seconds
Memory usage of 'execute': 1.2779 MB


In [7]:
res.resume

     feature group Chi2Test aa test TTest aa test KSTest aa test  \
0  cat_col_3  test           NOT OK           NaN            NaN   
1  num_col_1  test              NaN        NOT OK         NOT OK   

  TTest best split KSTest best split Chi2Test best split  result  difference  \
0              NaN               NaN                  OK  NOT OK         NaN   
1               OK                OK                 NaN  NOT OK      -0.758   

   difference %  
0           NaN  
1     -1.475914  